In [1]:
import json
from pathlib import Path
import sys
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

import spectra.data as data
from spectra.absorberspectrum import AbsorberSpectrum
from spectra.monomer import Monomer
from spectra.resin import ResinConstituentAbsorber, ResinConstituentMonomer, Resin

# Initial design thoughts

Requirements

- Specify absorber(s) and photoinitiator(s) - these can all just go into a list because all that matters is their absorption spectrum; need a way to specify their concentration (molar or w/w)
- Specify monomer(s) and their concentrations (molar or w/w), or **their fractions (100%, 50%-50%, or 25%-30%-45%, for example)**
- Calculate absorption coefficient as a function of wavelength. Should be able to interpolate values so can sample at any wavelength or set of wavelengths

Possible ways of specifying materials and concentrations

- List of tuples with tuples of the form: (material object, concentration)
- **List of objects with objects having attributes of material, concentration_w/w_perc**, concentration_molar
- List of tuples with tuples of the form: (material object, mass in resin formulation)

Resin class provides:

- Absorption coefficient as a function of wavelength
- Molar concentrations of all materials?
- Weight fractions of all materials?

# Resin constituent material classes

In [2]:
# @dataclass
# class ResinConstituentAbsorber:
#     """Container with 2 pieces of information:
#     1. Absorber material
#     2. Concentration w/w (percent) in resin
    
#     Examples 
#     --------
#     1% Irgacure 819:
#         `irgacure819 = ResinConstituentAbsorber(
#             data.photoinitiators['irgacure819_in_PEGDA_2020-01-20'],
#             1.0
#         )`
#     0.38% avobenzone:
#         `avobenzone = ResinConstituentAbsorber(
#             data.absorbers['avobenzone_in_PEGDA_2020-01-20'],
#             0.38
#         )`
#     """
#     material: AbsorberSpectrum
#     concentration_ww_percent: (int, float)
    
#     def __post_init__(self):
        
#         # Enforce data types
#         assert isinstance(self.material, AbsorberSpectrum)
#         assert isinstance(self.concentration_ww_percent, (int, float))

# @dataclass
# class ResinConstituentMonomer:
#     """Container with 2 pieces of information:
#     1. Monomer material
#     2. Concentration as percent of all MONOMERS in resin (not percent of total resin)
    
#     Example: monomers for 60-40 PEGDA-HDDA resin:
#         `pegda = ResinConstituentMonomer(data.monomers['PEGDA'], 60)`
#         `hdda = ResinConstituentMonomer(data.monomers['HDDA'], 40)`
#     """
#     material: Monomer
#     concentration_percent_monomer: (int, float)
    
#     def __post_init__(self):
        
#         # Enforce data types
#         assert isinstance(self.material, Monomer)
#         assert isinstance(self.concentration_percent_monomer, (int, float))


In [3]:
data.absorbers

{'avobenzone_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f8361ead750>,
 'avobenzone_in_PEGDA_2020-01-20': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f833047df10>,
 'benetex_obplus_in PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f83617c2290>,
 'bls99-2_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f8361ead950>,
 'martius_yellow_in PEGDA_2020-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f8361ead850>,
 'nps_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f8361eada50>,
 'octocrylene_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f8361ead910>,
 'phenazine_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f8361ec0dd0>,
 'salicylaldehyde_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f8361ec0e50>,
 'sudanI_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f8361ec08d0>}

In [4]:
temp_absorber = ResinConstituentAbsorber(
    data.absorbers["avobenzone_in_PEGDA_2020-01-20"],
    0.38
)
temp_absorber

ResinConstituentAbsorber(material=<spectra.absorberspectrum.AbsorberSpectrum object at 0x7f833047df10>, concentration_ww_percent=0.38)

In [19]:
temp_absorber.material

In [5]:
data.photoinitiators

{'irgacure819_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f8362151590>,
 'irgacure819_in_PEGDA_2020-01-20': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f8362151810>,
 'tmdpo_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f8362151390>}

In [18]:
temp_photoinitiator = ResinConstituentAbsorber(
    data.photoinitiators["irgacure819_in_PEGDA_2020-01-20"],
    1.0
)
temp_photoinitiator

ResinConstituentAbsorber(material=<spectra.absorberspectrum.AbsorberSpectrum object at 0x7f8362151810>, concentration_ww_percent=1.0)

In [7]:
data.monomers

{'HDDA': Monomer(/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/monomers/hdda.json),
 'PEGDA': Monomer(/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/monomers/pegda.json),
 'TET': Monomer(/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/monomers/tet.json)}

In [8]:
temp_monomer = ResinConstituentMonomer(
    data.monomers['PEGDA'],
    100
)
temp_monomer

ResinConstituentMonomer(material=Monomer(/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/monomers/pegda.json), concentration_percent_monomer=100)

In [9]:
vars(temp_monomer.material)

{'_json_file': PosixPath('/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/monomers/pegda.json'),
 '_from_json': {'Name': 'PEGDA',
  'Long name': 'poly ethylene glycol diacrylate',
  'Density g/L': 1110},
 'name': 'PEGDA',
 'long_name': 'poly ethylene glycol diacrylate',
 'density': 1110,
 'density_units': 'g/L'}

In [10]:
temp_monomer.material.density

1110

In [11]:
temp_monomer.material.density_units

'g/L'

# Resin class

In [12]:
# class Resin:
#     """Combine materials into a resin and calculate its absorption coefficient
#     as a function of wavelength.
    
#     Parameters
#     ----------
#     monomers - ResinConstituentMonomer of list of ResinConstituentMonomer
#     photoinitiators - ResinConstituentAbsorber or list of ResinConstituentAbsorber
#     absorbers - ResinConstituentAbsorber or list of ResinConstituentAbsorber
    
#     Examples
#     -------
#     PEGDA with 0.38% avobenzone and 1% Irgacure 819:
#         pegda = ResinConstituentMonomer(data.monomers['PEGDA'], 100)
#         avobenzone = ResinConstituentAbsorber(
#             data.absorbers['avobenzone_in_PEGDA_2020-01-20'],
#             0.38
#         )
#         irgacure819 = ResinConstituentAbsorber(
#             data.photoinitiators['irgacure819_in_PEGDA_2020-01-20'],
#             1.0
#         )
#         resin = Resin(monomers=pegda, absorbers=avobenzone, photoinitiators=irgacure819)
#     60-40 PEGDA-HDDA with 2% NPS, 1% avobenzone and 1% Irgacure 819:
#         pegda = ResinConstituentMonomer(data.monomers['PEGDA'], 60)
#         hdda = ResinConstituentMonomer(data.monomers['HDDA'], 40)
#         nps = ResinConstituentAbsorber(
#             data.absorbers['nps_in_PEGDA_2017-04-13'],
#             2.0
#         )
#         avobenzone = ResinConstituentAbsorber(
#             data.absorbers['avobenzone_in_PEGDA_2020-01-20'],
#             1.0
#         )
#         irgacure819 = ResinConstituentAbsorber(
#             data.photoinitiators['irgacure819_in_PEGDA_2020-01-20'],
#             1.0
#         )
#         resin = Resin(monomers=[pegda, hdda], absorbers=[nps, avobenzone], photoinitiators=irgacure819)
#     """
        
    
#     def __init__(self, *, monomers, absorbers=[], photoinitiators=[]):
        
#         # Handle inputs. Treat photoinitiators as absorbers for purpose of calculating absorption coefficient
#         self.monomers = self.create_list(monomers, ResinConstituentMonomer)
#         self.check_monomers(self.monomers)
#         self.absorbers = self.create_list(absorbers, ResinConstituentAbsorber) + \
#                          self.create_list(photoinitiators, ResinConstituentAbsorber)
#         if not self.absorbers:
#             raise ValueError("There must be at least one absorber or one photoinitiator.")
        
#         # Calculate molar concentration for absorbers, photoinitiators in list of absorbers
#         self._density = self.calc_density(self.monomers)
#         self._molar_concentrations = []
#         for a in self.absorbers:
#             self._molar_concentrations.append(
#                 (a.concentration_ww_percent / 100) * self._density / a.material.molar_mass_g_per_mole
#             )
        
#         # Set wavelength to be the same as for the first absorber material
#         self.wavelength = self.absorbers[0].material.wavelength.copy()
        
#         # Calculate absorption coefficient as a function of wavelength
#         self.absorption_coeff = np.zeros(self.absorbers[0].material.molar_absorptivity.shape)
#         for a in self.absorbers:
#             self.absorption_coeff += np.log(10) * \
#                 a.material.calc_molar_absorptivity(self.wavelength) * \
#                 (a.concentration_ww_percent/100) * \
#                 (self._density / a.material.molar_mass_g_per_mole)
        

#     def calc_absorption_coef_inv_cm(self, wavelength):
#         """Given wavelength, interpolate corresponding absorption coefficient value.
#         Units: cm^(-1)

#         See numpy.interp documentation for interpolation details.

#         Parameters
#         ----------
#         wavelength - array_like (can be single value or 1D list/numpy.array of values)

#         Returns
#         -------
#         single value or 1D numpy array corresponding to shape of wavelength
#         """
#         return np.interp(wavelength, self.wavelength, self.absorption_coeff)

        
#     @staticmethod
#     def create_list(item, desired_type):
        
#         if isinstance(item, desired_type):
#             return [item]
        
#         if isinstance(item, list):
#             for i in item:
#                 assert isinstance(i, desired_type)
#             return item
        
#         raise TypeError(f"Wrong type. Must be {desired_type} or list of {desired_type}.")
        
#     @staticmethod
#     def check_monomers(monomers):
#         '''Make sure percentages add to 100
#         '''
#         total = 0
#         for m in monomers:
#             total += m.concentration_percent_monomer
#         if total != 100:
#             raise ValueError("Sum of monomer concentrations must equal 100 (percent)")
        
#     @staticmethod
#     def calc_density(monomers):
#         weighted_average = 0
#         for m in monomers:
#             weighted_average += m.material.density * (m.concentration_percent_monomer/100.)
#         return weighted_average
        

## PEGDA-Avobenzone-Irgacure 819

In [13]:
pegda = ResinConstituentMonomer(data.monomers['PEGDA'], 100)
avobenzone = ResinConstituentAbsorber(
    data.absorbers['avobenzone_in_PEGDA_2020-01-20'],
    0.38
)
irgacure819 = ResinConstituentAbsorber(
    data.photoinitiators['irgacure819_in_PEGDA_2020-01-20'],
    1.0
)
resin = Resin(monomers=pegda, absorbers=avobenzone, photoinitiators=irgacure819)
resin

In [14]:
fig, ax = plt.subplots()
ax.plot(resin.wavelength, resin.absorption_coeff)
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Absorption coefficient (cm$^{-1}$)')
ax.set_title('PEGDA with 0.38% Avobenzone and 1% Irgacure 819')
ax.grid();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## No photoinitiator

In [15]:
resin_no_photoinitiator = Resin(monomers=pegda, absorbers=avobenzone)

fig, ax = plt.subplots()
ax.plot(resin_no_photoinitiator.wavelength, resin_no_photoinitiator.absorption_coeff)
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Absorption coefficient (cm$^{-1}$)')
ax.set_title('PEGDA with 0.38% Avobenzone')
ax.grid();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## No absorber

In [16]:
resin_no_absorber = Resin(monomers=pegda, photoinitiators=irgacure819)

fig, ax = plt.subplots()
ax.plot(resin_no_absorber.wavelength, resin_no_absorber.absorption_coeff)
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Absorption coefficient (cm$^{-1}$)')
ax.set_title('PEGDA with 1% Irgacure 819')
ax.grid();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## No absorber or photoinitiator

In [17]:
resin_no_absorber_or_photoinitiator = Resin(monomers=pegda)

ValueError: There must be at least one absorber or one photoinitiator.

## Resin name

In [2]:
data.material_abbreviations

{'PEGDA': 'PEG',
 'HDDA': 'HDD',
 'TET': 'TET',
 'Lauryl acrylate': 'Lau',
 'Avobenzone': 'Avo',
 'NPS': 'NPS',
 'Benetex OBplus': 'BOB',
 'BLS99-2': 'B99',
 'Martius Yellow': 'MaY',
 'Octocrylene': 'Oct',
 'Phenazine': 'Phe',
 'Salicylaldehyde': 'Sal',
 'Sudan I': 'SuI',
 'Irgacure 819': 'Irg',
 'TMDPO': 'TMD',
 'BME': 'BME'}

In [2]:
pegda = ResinConstituentMonomer(data.monomers['PEGDA'], 100)
avobenzone = ResinConstituentAbsorber(
    data.absorbers['avobenzone_in_PEGDA_2020-01-20'],
    0.38
)
irgacure819 = ResinConstituentAbsorber(
    data.photoinitiators['irgacure819_in_PEGDA_2020-01-20'],
    1.0
)
resin = Resin(monomers=pegda, absorbers=avobenzone, photoinitiators=irgacure819)

resin.name

'PEG-100__Avo-0.38__Irg-1'

In [3]:
hdda = ResinConstituentMonomer(data.monomers['PEGDA'], 85)
la = ResinConstituentMonomer(data.monomers['TET'], 15)
avobenzone = ResinConstituentAbsorber(
    data.absorbers['avobenzone_in_PEGDA_2020-01-20'],
    1.5
)
nps = ResinConstituentAbsorber(
    data.absorbers['nps_in_PEGDA_2017-04-13'],
    2.0
)
irgacure819 = ResinConstituentAbsorber(
    data.photoinitiators['irgacure819_in_PEGDA_2020-01-20'],
    1
)
tmdpo = ResinConstituentAbsorber(
    data.photoinitiators['tmdpo_in_PEGDA_2017-04-13'],
    0.25
)
resin = Resin(monomers=[hdda,la], absorbers=[avobenzone,nps], photoinitiators=[irgacure819,tmdpo])

resin.name

'PEG-85_TET-15__Avo-1.5_NPS-2__Irg-1_TMD-0.25'

In [18]:
temp = data.absorbers['avobenzone_in_PEGDA_2020-01-20']
temp.molar_absorptivity.shape

(306,)

In [19]:
temp_ma = np.zeros(temp.molar_absorptivity.shape)
temp_ma.shape

(306,)